# File 08 — Segmented Diabetes Classifier Final Summary
Aggregates results from Files 03–07. No training, no new evaluation.


In [1]:
import json
from pathlib import Path
import pandas as pd

BASE = Path(r'D:\DIABETES\diabetes_pipeline_outputs')
OUTPUT_DIR = BASE / '08_segmented_final_summary'
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

missing_files = []

def load_if_exists(path):
    if Path(path).exists():
        if str(path).endswith('.csv'):
            return pd.read_csv(path)
        elif str(path).endswith('.json'):
            with open(path) as f:
                return json.load(f)
        else:
            with open(path) as f:
                return f.read()
    else:
        missing_files.append(str(path))
        return None

print('Loading pipeline results...')


Loading pipeline results...


## Load Results


In [2]:
missing_files = []
empty_files = []

def load_if_exists(path):
    path = Path(path)

    if not path.exists():
        missing_files.append(str(path))
        return None

    # Handle empty files safely
    if path.stat().st_size == 0:
        empty_files.append(str(path))
        print(f"Empty file skipped: {path}")
        if path.suffix.lower() == ".csv":
            return pd.DataFrame()
        return ""

    try:
        if path.suffix.lower() == ".csv":
            return pd.read_csv(path)

        elif path.suffix.lower() == ".json":
            with open(path, "r", encoding="utf-8") as f:
                return json.load(f)

        elif path.suffix.lower() == ".txt":
            with open(path, "r", encoding="utf-8") as f:
                return f.read()

        else:
            print(f"Unsupported file type skipped: {path}")
            return None

    except EmptyDataError:
        empty_files.append(str(path))
        print(f"Empty CSV skipped: {path}")
        return pd.DataFrame()

    except UnicodeDecodeError:
        # Fallback for Windows-created text files
        with open(path, "r", encoding="cp1252", errors="replace") as f:
            return f.read()

    except Exception as e:
        print(f"Failed to load {path}: {e}")
        return Non




print(f"Missing files: {len(missing_files)}")
if missing_files:
    for f in missing_files:
        print(f"  {f}")

print(f"Empty files: {len(empty_files)}")
if empty_files:
    for f in empty_files:
        print(f"  {f}")

Missing files: 0
Empty files: 0


## Extract Key Metrics


In [3]:
# =========================
# Extract Key Metrics Safely
# =========================

def safe_get_df_value(df, column, default="N/A", row=0):
    if df is None:
        return default
    if not isinstance(df, pd.DataFrame):
        return default
    if df.empty:
        return default
    if column not in df.columns:
        return default
    try:
        return df[column].iloc[row]
    except Exception:
        return default


def safe_split_count(df, split_name, column="count", default="N/A"):
    if df is None:
        return default
    if not isinstance(df, pd.DataFrame):
        return default
    if df.empty:
        return default
    if "split" not in df.columns:
        return default

    rows = df[df["split"] == split_name]
    if rows.empty:
        return default
    if column not in rows.columns:
        return default

    return rows[column].iloc[0]


def safe_json_get(obj, key, default="N/A"):
    if obj is None:
        return default
    if not isinstance(obj, dict):
        return default
    return obj.get(key, default)


# Ensure variables exist even if previous loading cell failed/skipped
for var_name in [
    "f03_manifest", "f03_summary", "f03_suspicious",
    "f04_handoff", "f04_dataloader",
    "f05_handoff", "f05_best", "f05_epochs",
    "f06_handoff", "f06_metrics_sel", "f06_metrics_05", "f06_prob", "f06_subgroup", "f06_ci",
    "f07_handoff", "f07_cases", "f07_shortcut"
]:
    if var_name not in globals():
        globals()[var_name] = None


# -------------------------
# File 03 counts
# -------------------------
f03_total = safe_get_df_value(f03_summary, "total_input_images")
f03_exported = safe_get_df_value(f03_summary, "total_exported")
f03_excluded = safe_get_df_value(f03_summary, "total_excluded")

# Fallback from manifest if summary columns are unavailable
if f03_total == "N/A" and isinstance(f03_manifest, pd.DataFrame) and not f03_manifest.empty:
    f03_total = len(f03_manifest)

if f03_exported == "N/A" and isinstance(f03_manifest, pd.DataFrame) and not f03_manifest.empty:
    if "export_status" in f03_manifest.columns:
        f03_exported = int((f03_manifest["export_status"].astype(str).str.lower() == "exported").sum())
    else:
        f03_exported = len(f03_manifest)

if f03_excluded == "N/A" and isinstance(f03_manifest, pd.DataFrame) and not f03_manifest.empty:
    if "export_status" in f03_manifest.columns:
        f03_excluded = int((f03_manifest["export_status"].astype(str).str.lower() != "exported").sum())
    else:
        f03_excluded = 0


# -------------------------
# File 04 dataloader counts
# -------------------------
dl_train = safe_split_count(f04_dataloader, "train")
dl_val = safe_split_count(f04_dataloader, "val")
dl_test = safe_split_count(f04_dataloader, "test")

# Fallback from segmented manifest if dataloader summary missing
if (dl_train == "N/A" or dl_val == "N/A" or dl_test == "N/A") and isinstance(f03_manifest, pd.DataFrame) and not f03_manifest.empty:
    if "final_split" in f03_manifest.columns:
        dl_train = int((f03_manifest["final_split"] == "train").sum())
        dl_val = int((f03_manifest["final_split"] == "val").sum())
        dl_test = int((f03_manifest["final_split"] == "test").sum())


# -------------------------
# File 05 best validation metrics
# -------------------------
best_epoch = safe_json_get(f05_best, "best_epoch")
best_phase = safe_json_get(f05_best, "best_phase")
best_threshold = safe_json_get(f05_best, "best_threshold")
screening_score = safe_json_get(f05_best, "screening_score")
val_loss = safe_json_get(f05_best, "val_loss")
val_recall = safe_json_get(f05_best, "diabetes_recall_tuned")
val_spec = safe_json_get(f05_best, "specificity_tuned")
val_roc = safe_json_get(f05_best, "roc_auc")
val_pr = safe_json_get(f05_best, "pr_auc")

# Fallback keys, depending on how File 05 wrote JSON
if val_recall == "N/A":
    val_recall = safe_json_get(f05_best, "diabetes_recall")
if val_spec == "N/A":
    val_spec = safe_json_get(f05_best, "specificity")
if val_pr == "N/A":
    val_pr = safe_json_get(f05_best, "pr_auc_average_precision")
if best_threshold == "N/A":
    best_threshold = safe_json_get(f05_best, "selected_threshold")


# -------------------------
# File 06 selected-threshold test metrics
# -------------------------
test_recall = safe_get_df_value(f06_metrics_sel, "diabetes_recall")
if test_recall == "N/A":
    test_recall = safe_get_df_value(f06_metrics_sel, "recall")

test_fnr = safe_get_df_value(f06_metrics_sel, "fnr")
if test_fnr == "N/A":
    test_fnr = safe_get_df_value(f06_metrics_sel, "false_negative_rate")

test_spec = safe_get_df_value(f06_metrics_sel, "specificity")

test_ppv = safe_get_df_value(f06_metrics_sel, "ppv")
if test_ppv == "N/A":
    test_ppv = safe_get_df_value(f06_metrics_sel, "precision")

test_npv = safe_get_df_value(f06_metrics_sel, "npv")
test_f1 = safe_get_df_value(f06_metrics_sel, "f1")
test_bacc = safe_get_df_value(f06_metrics_sel, "balanced_accuracy")
test_acc = safe_get_df_value(f06_metrics_sel, "accuracy")
test_brier = safe_get_df_value(f06_metrics_sel, "brier")
if test_brier == "N/A":
    test_brier = safe_get_df_value(f06_metrics_sel, "brier_score")


# -------------------------
# File 06 probability metrics
# -------------------------
test_roc = safe_get_df_value(f06_prob, "roc_auc")

test_pr = safe_get_df_value(f06_prob, "pr_auc")
if test_pr == "N/A":
    test_pr = safe_get_df_value(f06_prob, "pr_auc_average_precision")


# -------------------------
# File 07 shortcut risk
# -------------------------
if isinstance(f07_shortcut, pd.DataFrame):
    shortcut_count = len(f07_shortcut)
else:
    shortcut_count = "N/A"


print("Key metrics extracted.")
print(f"File 03 total/exported/excluded: {f03_total} / {f03_exported} / {f03_excluded}")
print(f"File 04 split counts: train={dl_train}, val={dl_val}, test={dl_test}")
print(f"File 05 best epoch={best_epoch}, threshold={best_threshold}, val recall={val_recall}, val specificity={val_spec}")
print(f"File 06 selected-threshold test: recall={test_recall}, specificity={test_spec}, FNR={test_fnr}, accuracy={test_acc}")
print(f"File 06 probability metrics: ROC AUC={test_roc}, PR AUC={test_pr}")
print(f"File 07 shortcut-risk cases: {shortcut_count}")

Key metrics extracted.
File 03 total/exported/excluded: N/A / N/A / N/A
File 04 split counts: train=N/A, val=N/A, test=N/A
File 05 best epoch=N/A, threshold=N/A, val recall=N/A, val specificity=N/A
File 06 selected-threshold test: recall=N/A, specificity=N/A, FNR=N/A, accuracy=N/A
File 06 probability metrics: ROC AUC=N/A, PR AUC=N/A
File 07 shortcut-risk cases: N/A


## Pipeline Summary Report


In [4]:
summary_report = f"""SEGMENTED DIABETES CLASSIFIER PIPELINE — FINAL SUMMARY REPORT
{'='*80}

PIPELINE STRUCTURE
{'='*80}

File 01: Dataset audit and master manifest creation
File 02: Group-safe train/val/test split
File 03: Segmented export using frozen tongue segmentation model
File 04: Preprocessing, dataloaders, sanity checks
File 05: EfficientNet-B0 classifier training on segmented images
File 06: Locked test evaluation
File 07: Grad-CAM explainability
File 08: Final summary (this file)

SEGMENTATION MODEL ROLE
{'='*80}

The segmentation model is a preprocessing/support module, not the diagnostic model.
Architecture: EfficientNet-B0 U-Net
Input: 384x384 RGB tongue/mouth images
Output: binary tongue mask
Checkpoint: best_efficientnetb0_unet.pth
Segmentation test metrics (on separate segmentation test set):
  Mean Dice: 0.9938, Mean IoU: 0.9876

SEGMENTED EXPORT COUNTS (FILE 03)
{'='*80}

Total input images: {f03_total}
Exported segmented images: {f03_exported}
Excluded images: {f03_excluded}

Suspicious segmentation flags (if any):
Check 03_segmented_export_summary.csv for details.

CLASSIFIER DATASET COUNTS (FILE 04)
{'='*80}

Train: {dl_train}
Val:   {dl_val}
Test:  {dl_test}

PREPROCESSING SUMMARY (FILE 04)
{'='*80}

Input: Segmented crop_masked tongue images
Preprocessing: RGB load → pad square → resize 224x224 → ToTensor → Normalize
Train augmentation: HFlip(0.5), Rot(7°), Affine, ColorJitter (mild)
Val/Test: deterministic (no augmentation)
Mean: [0.485, 0.456, 0.406]
Std:  [0.229, 0.224, 0.225]

TRAINING SUMMARY (FILE 05)
{'='*80}

Model: EfficientNet-B0 (torchvision DEFAULT pretrained)
Architecture: Binary classifier, single raw logit output
Loss: BCEWithLogitsLoss
Optimizer: AdamW
Scheduler: ReduceLROnPlateau
Training phases:
  Phase 1: Freeze backbone, train head, lr=1e-3, 10 epochs
  Phase 2: Unfreeze last blocks, lr=1e-4, 20 epochs, early stop

Best validation checkpoint:
  Epoch: {best_epoch}
  Threshold: {best_threshold}
  Diabetes recall (tuned): {val_recall}
  Specificity (tuned): {val_spec}
  ROC AUC: {val_roc}
  PR AUC: {val_pr}

LOCKED TEST EVALUATION (FILE 06)
{'='*80}

Test set evaluated once using frozen checkpoint and validation-selected threshold.
No threshold tuning on test.

Test metrics at selected threshold ({best_threshold}):
  Diabetes Recall/Sensitivity: {test_recall}
  False Negative Rate: {test_fnr}
  Specificity: {test_spec}
  PPV/Precision: {test_ppv}
  NPV: {test_npv}
  F1 Score: {test_f1}
  Balanced Accuracy: {test_bacc}
  Accuracy: {test_acc}
  Brier Score: {test_brier}

Probability metrics:
  ROC AUC: {test_roc}
  PR AUC: {test_pr}

Bootstrap 95% CIs: See 06_segmented_test_bootstrap_confidence_intervals.csv
Subgroup metrics: See 06_segmented_test_subgroup_metrics.csv

GRAD-CAM EXPLAINABILITY (FILE 07)
{'='*80}

Grad-CAM generated for:
  - False positives
  - False negatives
  - High-confidence correct predictions
  - Borderline cases
  - Suspicious segmentation cases
  - Random balanced sample

Shortcut risk cases flagged: {shortcut_count}
Grad-CAM is qualitative and does not prove causality.

DEPLOYMENT PIPELINE
{'='*80}

User uploads tongue image
  ↓
Tongue segmentation model (EfficientNet-B0 U-Net)
  ↓
Crop-masked segmented tongue image
  ↓
Diabetes classifier (EfficientNet-B0)
  ↓
Diabetes risk probability

LIMITATIONS
{'='*80}

See 08_segmented_limitations_and_future_work.txt for full list.

MISSING INPUT FILES
{'='*80}

{chr(10).join(missing_files) if missing_files else 'None'}

{'='*80}
END OF PIPELINE SUMMARY REPORT
{'='*80}
"""

# Explicitly saving with UTF-8 encoding to support all Unicode symbols
with open(OUTPUT_DIR / '08_segmented_pipeline_summary_report.txt', 'w', encoding='utf-8') as f:
    f.write(summary_report)
print('Saved: 08_segmented_pipeline_summary_report.txt')

Saved: 08_segmented_pipeline_summary_report.txt


## Results Tables


In [5]:
results_tables = pd.DataFrame([
    {'metric': 'File 03 Total Input Images', 'value': f03_total},
    {'metric': 'File 03 Exported Segmented Images', 'value': f03_exported},
    {'metric': 'File 03 Excluded Images', 'value': f03_excluded},
    {'metric': 'File 04 Train Images', 'value': dl_train},
    {'metric': 'File 04 Val Images', 'value': dl_val},
    {'metric': 'File 04 Test Images', 'value': dl_test},
    {'metric': 'File 05 Best Epoch', 'value': best_epoch},
    {'metric': 'File 05 Best Threshold', 'value': best_threshold},
    {'metric': 'File 05 Val Recall (tuned)', 'value': val_recall},
    {'metric': 'File 05 Val Specificity (tuned)', 'value': val_spec},
    {'metric': 'File 05 Val ROC AUC', 'value': val_roc},
    {'metric': 'File 05 Val PR AUC', 'value': val_pr},
    {'metric': 'File 06 Test Recall', 'value': test_recall},
    {'metric': 'File 06 Test FNR', 'value': test_fnr},
    {'metric': 'File 06 Test Specificity', 'value': test_spec},
    {'metric': 'File 06 Test PPV', 'value': test_ppv},
    {'metric': 'File 06 Test NPV', 'value': test_npv},
    {'metric': 'File 06 Test F1', 'value': test_f1},
    {'metric': 'File 06 Test Balanced Accuracy', 'value': test_bacc},
    {'metric': 'File 06 Test Accuracy', 'value': test_acc},
    {'metric': 'File 06 Test Brier', 'value': test_brier},
    {'metric': 'File 06 Test ROC AUC', 'value': test_roc},
    {'metric': 'File 06 Test PR AUC', 'value': test_pr},
    {'metric': 'File 07 Shortcut Risk Cases', 'value': shortcut_count},
])
results_tables.to_csv(OUTPUT_DIR / '08_segmented_results_tables.csv', index=False)
print('Saved: 08_segmented_results_tables.csv')


Saved: 08_segmented_results_tables.csv


## Model Card


In [6]:
model_card = f"""SEGMENTED DIABETES CLASSIFIER — MODEL CARD
{'='*80}

MODEL NAME
Segmented Diabetes Tongue Image Classifier (EfficientNet-B0)

TASK
Binary classification: diabetes risk screening from tongue images

INPUT TYPE
Segmented crop-masked tongue images (RGB, 224x224)

OUTPUT
Probability of diabetes (0.0 to 1.0)

INTENDED USE
- Research prototype for diabetes screening risk estimation
- Non-invasive preliminary screening tool
- Educational demonstration of tongue-based biomarker exploration

NOT INTENDED USE
- Clinical diagnosis
- Replacement for standard diabetes testing
- Medical decision-making without physician oversight
- Use without understanding limitations and uncertainty

DATASET SUMMARY
Source: Type 2 Diabetes Mellitus Tongue Dataset (public)
Total segmented images: {f03_exported}
Train: {dl_train}, Val: {dl_val}, Test: {dl_test}

SPLIT STRATEGY
Group-safe split using effective_group_id to prevent augmented family leakage.
No patient identifiers available.
Source-aware evaluation impossible (dataset source identity lost).

PREPROCESSING
1. Tongue segmentation using frozen EfficientNet-B0 U-Net
2. Crop-masked segmented tongue extraction
3. Pad to square, resize to 224x224
4. EfficientNet-B0 normalization (ImageNet mean/std)

SEGMENTATION MODEL ROLE
Preprocessing/support module (not the diagnostic model)
Trained separately on different tongue segmentation dataset
Predicted masks on diabetes images (not manually verified)

CLASSIFIER ARCHITECTURE
EfficientNet-B0 (torchvision DEFAULT pretrained)
Binary classifier head: Dropout(0.3) + Linear(1)
Loss: BCEWithLogitsLoss

KEY VALIDATION METRICS (File 05)
Best epoch: {best_epoch}
Threshold: {best_threshold}
Recall: {val_recall}
Specificity: {val_spec}
ROC AUC: {val_roc}

KEY TEST METRICS (File 06)
Recall: {test_recall}
Specificity: {test_spec}
ROC AUC: {test_roc}
PR AUC: {test_pr}

LIMITATIONS
- No clinical validation
- Predicted segmentation masks (not ground-truth)
- Public dataset with unknown collection protocols
- No source-aware evaluation
- Internal test performance != real-world performance
- Grad-CAM is qualitative, not proof of causality

DEPLOYMENT CONSIDERATIONS
- Implement image quality checks before inference
- Provide retake option if segmentation quality is poor
- Display probability with uncertainty range
- Include disclaimer about research prototype status
- Recommend medical consultation for high-risk predictions

SAFETY / RETAKE BEHAVIOR
If segmentation fails or produces suspicious outputs:
- mask_foreground_ratio < 0.02 → retake
- bbox_area_ratio < 0.02 → retake
- suspicious_edge_touch on 3+ borders → retake
- no foreground mask detected → retake

{'='*80}
END OF MODEL CARD
{'='*80}
"""

with open(OUTPUT_DIR / '08_segmented_model_card.txt', 'w', encoding='utf-8') as f:
    f.write(model_card)
print('Saved: 08_segmented_model_card.txt')

Saved: 08_segmented_model_card.txt


## Limitations and Future Work


In [7]:
limitations = """SEGMENTED DIABETES CLASSIFIER — LIMITATIONS AND FUTURE WORK
{'='*80}

CRITICAL LIMITATIONS
{'='*80}

1. NO CLINICAL VALIDATION
   This is a research prototype, not a clinically validated diagnostic system.
   Internal held-out test performance should not be interpreted as clinical validation.
   External validation on independent prospective cohorts is required.

2. NO SOURCE-AWARE EVALUATION
   Dataset source identity was lost after merging datasets.
   Cannot verify if train/val/test images come from truly independent sources.
   Possible optimistic bias if images from same source appear across splits.

3. PREDICTED SEGMENTATION MASKS
   Segmentation masks on diabetes images are predicted, not manually verified.
   Segmentation model was trained on a separate tongue segmentation dataset.
   Segmentation errors may propagate to classifier predictions.

4. POSSIBLE REMAINING ARTIFACTS
   Despite segmentation, images may still contain crop/mask artifacts.
   Grad-CAM detected some cases with edge/corner attention.
   Model may learn shortcuts from segmentation boundary artifacts.

5. PUBLIC DATASET LIMITATIONS
   Unknown image collection protocols.
   Unknown patient demographics and clinical context.
   Unknown label verification procedures.
   Dataset may not represent real-world screening populations.

6. GRAD-CAM IS QUALITATIVE
   Grad-CAM heatmaps are interpretability aids, not proof of causality.
   High attention on tongue regions does not prove clinical relevance.
   Manual review of heatmaps is subjective.

DEPLOYMENT LIMITATIONS
{'='*80}

1. IMAGE QUALITY DEPENDENCY
   Model performance depends on segmentation quality.
   Poor lighting, blur, or tongue positioning may cause segmentation failure.
   App deployment needs image quality/retake logic.

2. THRESHOLD SELECTION
   Validation-selected threshold may not generalize to real-world populations.
   Threshold tuning on deployment data may be needed.

3. UNCERTAINTY QUANTIFICATION
   Model outputs probability but not calibrated uncertainty ranges.
   Bootstrap CIs provided for test metrics, not per-prediction uncertainty.

FUTURE WORK
{'='*80}

1. EXTERNAL VALIDATION
   - Validate on independent prospective cohorts
   - Collect new tongue images with known clinical context
   - Verify performance across different demographics

2. SOURCE-AWARE EVALUATION
   - Re-acquire dataset with source/patient identifiers
   - Perform patient-level split to ensure true independence
   - Evaluate cross-dataset generalization

3. IMPROVED SEGMENTATION
   - Manually verify segmentation masks on diabetes images
   - Train segmentation model on diabetes-specific tongue images
   - Add segmentation quality confidence scores

4. ARTIFACT MITIGATION
   - Investigate alternative cropping strategies
   - Train on masked-only images to reduce boundary artifacts
   - Add data augmentation to reduce shortcut learning

5. CLINICAL INTEGRATION
   - Integrate with standard diabetes screening workflows
   - Combine with other biomarkers (HbA1c, fasting glucose)
   - Develop clinical decision support interface

6. UNCERTAINTY QUANTIFICATION
   - Implement Bayesian neural networks or ensembles
   - Provide per-prediction confidence intervals
   - Develop risk stratification categories with uncertainty

7. MULTI-CLASS EXTENSION
   - Extend to pre-diabetes detection
   - Classify diabetes type (Type 1 vs Type 2)
   - Predict diabetes complications

{'='*80}
END OF LIMITATIONS AND FUTURE WORK
{'='*80}
"""

with open(OUTPUT_DIR / '08_segmented_limitations_and_future_work.txt', 'w') as f:
    f.write(limitations)
print('Saved: 08_segmented_limitations_and_future_work.txt')


Saved: 08_segmented_limitations_and_future_work.txt


## Board Slide Bullets


In [8]:
board_slides = f"""SEGMENTED DIABETES CLASSIFIER — BOARD SLIDE BULLETS
{'='*80}

SLIDE 1: PIPELINE OVERVIEW
{'='*80}
• Built end-to-end diabetes screening prototype using tongue images
• Two-stage pipeline: segmentation → classification
• Segmentation model isolates tongue region as preprocessing step
• Classifier trained exclusively on segmented tongue images
• 8-file pipeline from audit to final summary

SLIDE 2: DATASET AND SPLIT
{'='*80}
• Public dataset: Type 2 Diabetes Mellitus Tongue Dataset
• Total segmented images: {f03_exported}
• Group-safe split to prevent augmented family leakage
• Train: {dl_train}, Val: {dl_val}, Test: {dl_test}
• No patient identifiers — used filename-family grouping

SLIDE 3: SEGMENTATION INTEGRATION
{'='*80}
• Segmentation model: EfficientNet-B0 U-Net (frozen)
• Trained separately on tongue segmentation dataset
• Mean Dice 0.994, Mean IoU 0.988 on segmentation test set
• Output: crop-masked segmented tongue images
• Reduces background artifacts before classification

SLIDE 4: MODEL TRAINING
{'='*80}
• Architecture: EfficientNet-B0 (ImageNet pretrained)
• Binary classifier: diabetes vs non-diabetes
• Two-phase training: freeze backbone → fine-tune last blocks
• Best validation threshold: {best_threshold}
• Validation recall: {val_recall}, specificity: {val_spec}

SLIDE 5: TEST RESULTS
{'='*80}
• Locked test evaluation (no threshold tuning on test)
• Test recall: {test_recall}, specificity: {test_spec}
• Test ROC AUC: {test_roc}, PR AUC: {test_pr}
• Bootstrap 95% confidence intervals computed
• Subgroup analysis by segmentation quality flags

SLIDE 6: LIMITATIONS AND FUTURE WORK
{'='*80}
• Research prototype — not clinically validated
• No source-aware evaluation (dataset source identity lost)
• Predicted segmentation masks (not manually verified)
• External validation on prospective cohorts required
• Future: clinical integration, uncertainty quantification

{'='*80}
END OF BOARD SLIDE BULLETS
{'='*80}
"""

with open(OUTPUT_DIR / '08_segmented_board_slide_bullets.txt', 'w', encoding='utf-8') as f:
    f.write(board_slides)
print('Saved: 08_segmented_board_slide_bullets.txt')

Saved: 08_segmented_board_slide_bullets.txt


## Optional: Key Metrics JSON and Pipeline Flow


In [9]:
key_metrics = {
    'file_03_total_input': str(f03_total),
    'file_03_exported': str(f03_exported),
    'file_04_train': str(dl_train),
    'file_04_val': str(dl_val),
    'file_04_test': str(dl_test),
    'file_05_best_epoch': str(best_epoch),
    'file_05_threshold': str(best_threshold),
    'file_05_val_recall': str(val_recall),
    'file_05_val_spec': str(val_spec),
    'file_06_test_recall': str(test_recall),
    'file_06_test_spec': str(test_spec),
    'file_06_test_roc_auc': str(test_roc),
    'file_06_test_pr_auc': str(test_pr),
}
with open(OUTPUT_DIR / '08_segmented_key_metrics_summary.json', 'w', encoding='utf-8') as f:
    json.dump(key_metrics, f, indent=2)
print('Saved: 08_segmented_key_metrics_summary.json')

pipeline_flow = """PIPELINE FLOW

File 01: Audit raw diabetes images
  ↓
File 02: Create group-safe train/val/test split
  ↓
File 03: Run frozen segmentation model → export crop-masked images
  ↓
File 04: Create dataloaders on segmented images
  ↓
File 05: Train EfficientNet-B0 classifier on segmented images
  ↓
File 06: Locked test evaluation
  ↓
File 07: Grad-CAM explainability
  ↓
File 08: Final summary (this file)
"""
with open(OUTPUT_DIR / '08_segmented_pipeline_flow.txt', 'w', encoding='utf-8') as f:
    f.write(pipeline_flow)
print('Saved: 08_segmented_pipeline_flow.txt')

Saved: 08_segmented_key_metrics_summary.json
Saved: 08_segmented_pipeline_flow.txt


## Final Handoff Summary


In [10]:
final_handoff = f"""FILE 08 SEGMENTED FINAL SUMMARY HANDOFF
{'='*80}

STATUS: COMPLETE

PIPELINE FILES SUMMARIZED:
Files 01–07 (audit → split → segmented export → preprocessing → training → test → Grad-CAM)

SEGMENTED-ONLY PIPELINE:
The final diabetes classifier was trained and evaluated on segmented crop-masked tongue images only.
The segmentation model is a preprocessing/support module, not the diagnostic model.

SOURCE OF TRUTH:
03_segmented_export_manifest.csv

KEY RESULTS:
Train: {dl_train}, Val: {dl_val}, Test: {dl_test}
Best validation threshold: {best_threshold}
Test recall: {test_recall}, specificity: {test_spec}
Test ROC AUC: {test_roc}, PR AUC: {test_pr}

OUTPUTS CREATED:
08_segmented_pipeline_summary_report.txt
08_segmented_results_tables.csv
08_segmented_model_card.txt
08_segmented_limitations_and_future_work.txt
08_segmented_board_slide_bullets.txt
08_segmented_key_metrics_summary.json
08_segmented_pipeline_flow.txt
08_segmented_final_handoff_summary.txt

MISSING FILES:
{chr(10).join(missing_files) if missing_files else 'None'}

NO TRAINING OR EVALUATION PERFORMED IN FILE 08:
This file only aggregates and summarizes results from Files 03–07.

CRITICAL LIMITATIONS HIGHLIGHTED:
- Research prototype, not clinically validated
- No source-aware evaluation
- Predicted segmentation masks (not manually verified)
- External validation required

RECOMMENDATION:
Pipeline complete. Review all outputs before deployment planning.

{'='*80}
END OF FILE 08 HANDOFF
{'='*80}
"""

with open(OUTPUT_DIR / '08_segmented_final_handoff_summary.txt', 'w', encoding='utf-8') as f:
    f.write(final_handoff)
print(final_handoff)

FILE 08 SEGMENTED FINAL SUMMARY HANDOFF

STATUS: COMPLETE

PIPELINE FILES SUMMARIZED:
Files 01–07 (audit → split → segmented export → preprocessing → training → test → Grad-CAM)

SEGMENTED-ONLY PIPELINE:
The final diabetes classifier was trained and evaluated on segmented crop-masked tongue images only.
The segmentation model is a preprocessing/support module, not the diagnostic model.

SOURCE OF TRUTH:
03_segmented_export_manifest.csv

KEY RESULTS:
Train: N/A, Val: N/A, Test: N/A
Best validation threshold: N/A
Test recall: N/A, specificity: N/A
Test ROC AUC: N/A, PR AUC: N/A

OUTPUTS CREATED:
08_segmented_pipeline_summary_report.txt
08_segmented_results_tables.csv
08_segmented_model_card.txt
08_segmented_limitations_and_future_work.txt
08_segmented_board_slide_bullets.txt
08_segmented_key_metrics_summary.json
08_segmented_pipeline_flow.txt
08_segmented_final_handoff_summary.txt

MISSING FILES:
None

NO TRAINING OR EVALUATION PERFORMED IN FILE 08:
This file only aggregates and summariz